In [ ]:
# -------------------------------------------------------------------
# 0.1-year alpha/beta boundary sensitivity table
# -------------------------------------------------------------------

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import WDM

from few.waveform import GenerateEMRIWaveform
from few.utils.constants import YRSID_SI

from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import generate_windowed_padded_waveform

from WDM_LISAresponse.code.lisa_response.TDI import (
    precompute_gen2_michelson_td_geometry,
    precompute_gen2_michelson_td_waveform,
    gen2_michelson_td_from_precomputed,
    gen2_michelson_wdm,
)

from WDM_LISAresponse.code.utils.utils import (
    compute_wdm_cut_info,
    apply_wdm_cut,
    wdm_mismatch,
)

# -------------------------------------------------------------------
# Output folder
# -------------------------------------------------------------------

RUN_NAME = "alpha_beta_boundary_sensitivity_0p1yr"
OUTPUT_DIR = Path("runs") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Orbit file
# -------------------------------------------------------------------

ORBIT_FILE = "esa-trailing-orbits.h5"

# -------------------------------------------------------------------
# Time-series / WDM settings
# -------------------------------------------------------------------

DURATION_YR = 0.1
dt = 10.0

Nf = 500
d = 4
A_frac = 0.25
q_wdm = 4

CALC_M0 = False
L_trunc_value = 25

# -------------------------------------------------------------------
# Sweep values
# -------------------------------------------------------------------

alpha_values = [0.0025, 0.005, 0.01]
beta_values = [2.0, 4.0]   # beta = tukey_factor

# -------------------------------------------------------------------
# TD reference settings
# -------------------------------------------------------------------

TD_NUM_INT_POINTS = 20
TD_SHIFT_ORDER = 7
TD_SHIFT_MODE = "zero"
TD_CHUNK_SIZE = 100_000

# -------------------------------------------------------------------
# Fast WDM settings
# -------------------------------------------------------------------

METHOD_LABEL = "lagblock_L25_R2048_B32_ip16_no_m0"

FAST_ONE_ARM_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "row_chunk_size": 2048,
    "lag_block_size": 32,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 16,
    "tl_tp_interp_kind": "linear",
    "tl_tp_interp_backend": "auto",
}

FAST_TDI_KWARGS = {
    **FAST_ONE_ARM_KWARGS,
    "batch_chunk": 1,
}

FAST_TDI_ASSEMBLY_MODE = "batch_map"

print("Output directory:", OUTPUT_DIR)
print("DURATION_YR:", DURATION_YR)
print("Nf:", Nf)
print("dt:", dt)
print("Delta T:", Nf * dt, "s")
print("alpha values:", alpha_values)
print("beta values:", beta_values)

Output directory: runs/alpha_beta_boundary_sensitivity_0p1yr
DURATION_YR: 0.1
Nf: 500
dt: 10.0
Delta T: 5000.0 s
alpha values: [0.0025, 0.005, 0.01]
beta values: [2.0, 4.0]


In [3]:
# -------------------------------------------------------------------
# FEW waveform generator
# -------------------------------------------------------------------

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

# -------------------------------------------------------------------
# LISA trailing orbits
# -------------------------------------------------------------------

orbit_data = read_orbit_file(ORBIT_FILE)

orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)

orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0_offset_yr = 1.0
t0 = orbit_data.t0 + t0_offset_yr * YRSID_SI

print("Orbit file:", ORBIT_FILE)
print("t0:", t0)

# -------------------------------------------------------------------
# EMRI parameters
# -------------------------------------------------------------------

m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0

dist = 1.0

theta = np.pi / 2
phi = 0.0

qK = np.pi / 3
phiK = np.pi / 3

Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3

eps = 1e-2

model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

print("EMRI model kwargs ready.")

Orbit file: esa-trailing-orbits.h5
t0: 2104769280.5810456
EMRI model kwargs ready.


In [4]:
# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------

def block_until_ready(x):
    if hasattr(x, "block_until_ready"):
        x.block_until_ready()
    return x


def materialize_xyz(xyz):
    out = []
    for arr in xyz:
        block_until_ready(arr)
        out.append(np.asarray(arr))
    return tuple(out)


def flatten_xyz(X, Y, Z):
    return np.concatenate([
        np.ravel(np.asarray(X)),
        np.ravel(np.asarray(Y)),
        np.ravel(np.asarray(Z)),
    ])


def xyz_mismatch(X_ref, Y_ref, Z_ref, X_app, Y_app, Z_app):
    ref = flatten_xyz(X_ref, Y_ref, Z_ref)
    app = flatten_xyz(X_app, Y_app, Z_app)
    return wdm_mismatch(ref, app)


def make_waveform(alpha, beta):
    times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
        waveform_model="emri",
        t0=t0,
        dt=dt,
        T_obs=DURATION_YR,
        model_kwargs=model_kwargs,
        window_function="tukey",
        window_kwargs={
            "alpha": float(alpha),
            "tukey_factor": float(beta),
        },
        wdm_Nf=Nf,
        return_padding_info=True,
    )

    return times_pad, hplus_td, hcross_td, pad_info

In [5]:
# -------------------------------------------------------------------
# Run one alpha/beta case
# -------------------------------------------------------------------

def run_alpha_beta_case(alpha, beta):
    print()
    print("=" * 72)
    print(f"Running alpha={alpha}, beta={beta}")
    print("=" * 72)

    # Fresh caches per case, to avoid memory growth.
    tdi_shift_cache = {}
    td_one_arm_cache = {}
    wdm_one_arm_cache = {}

    # ------------------------------------------------------------
    # 1. Waveform
    # ------------------------------------------------------------

    times_pad, hplus_td, hcross_td, pad_info = make_waveform(alpha, beta)

    N_original = int(pad_info["original_length"])

    print("Waveform generated:")
    print("  original length:", N_original)
    print("  padded length:  ", len(times_pad))
    print("  WDM Nt:         ", pad_info["wdm_Nt"])

    # ------------------------------------------------------------
    # 2. TD Gen-2 Michelson reference
    # ------------------------------------------------------------

    td_geom_precomp = precompute_gen2_michelson_td_geometry(
        times_pad,
        theta,
        phi,
        orbits_trailing,
        shifts_cache=tdi_shift_cache,
    )

    td_wave_precomp = precompute_gen2_michelson_td_waveform(
        td_geom_precomp,
        hplus_td,
        hcross_td,
        num_int_points=TD_NUM_INT_POINTS,
        one_arm_cache=td_one_arm_cache,
        chunk_size=TD_CHUNK_SIZE,
    )

    X_td, Y_td, Z_td = gen2_michelson_td_from_precomputed(
        td_wave_precomp,
        order=TD_SHIFT_ORDER,
        mode=TD_SHIFT_MODE,
    )

    X_td = np.asarray(X_td)
    Y_td = np.asarray(Y_td)
    Z_td = np.asarray(Z_td)

    # ------------------------------------------------------------
    # 3. Direct WDM Gen-2 Michelson response
    # ------------------------------------------------------------

    X_wdm, Y_wdm, Z_wdm = materialize_xyz(
        gen2_michelson_wdm(
            times_pad,
            hplus_td,
            hcross_td,
            theta,
            phi,
            orbits_trailing,
            Nf=Nf,
            d=d,
            q=q_wdm,
            calc_m0=CALC_M0,
            shifts_cache=tdi_shift_cache,
            L_trunc=L_trunc_value,
            one_arm_cache=wdm_one_arm_cache,
            variable_shift_kwargs=FAST_TDI_KWARGS,
            use_jax=True,
            tdi_assembly_mode=FAST_TDI_ASSEMBLY_MODE,
        )
    )

    X_wdm = np.real(np.asarray(X_wdm))
    Y_wdm = np.real(np.asarray(Y_wdm))
    Z_wdm = np.real(np.asarray(Z_wdm))

    # ------------------------------------------------------------
    # 4. Transform TD reference into WDM domain
    # ------------------------------------------------------------

    wdm_ref = WDM.WDM.WDM_transform(
        dt=dt,
        Nf=Nf,
        N=len(times_pad),
        q=q_wdm,
        d=d,
        calc_m0=CALC_M0,
        A_frac=A_frac,
    )

    X_td_w = wdm_ref.dwt(X_td)
    Y_td_w = wdm_ref.dwt(Y_td)
    Z_td_w = wdm_ref.dwt(Z_td)

    if X_td_w.shape != X_wdm.shape:
        raise RuntimeError(f"Shape mismatch: X_td_w={X_td_w.shape}, X_wdm={X_wdm.shape}")

    # ------------------------------------------------------------
    # 5. Boundary cut
    # ------------------------------------------------------------

    cut_info = compute_wdm_cut_info(
        original_length=N_original,
        Nf=Nf,
        dt=dt,
        alpha=float(alpha),
        tukey_factor=float(beta),
        window_pad_left=pad_info["window_pad_left"],
        window_pad_right=pad_info["window_pad_right"],
        include_wdm_tail_padding=True,
        include_taper_margin=True,
        wdm_dT=wdm_ref.dT,
        wdm_Nt=wdm_ref.Nt,
        drop_m0=False,
        drop_freq_edges=False,
    )

    left_cut = int(cut_info["left_cut"])
    right_cut = int(cut_info["right_cut"])
    rows_removed = left_cut + right_cut

    # ------------------------------------------------------------
    # 6. Uncut and cut combined XYZ mismatch
    # ------------------------------------------------------------

    overlap_uncut, mismatch_uncut = xyz_mismatch(
        X_td_w, Y_td_w, Z_td_w,
        X_wdm, Y_wdm, Z_wdm,
    )

    X_td_cut = apply_wdm_cut(X_td_w, cut_info)
    Y_td_cut = apply_wdm_cut(Y_td_w, cut_info)
    Z_td_cut = apply_wdm_cut(Z_td_w, cut_info)

    X_wdm_cut = apply_wdm_cut(X_wdm, cut_info)
    Y_wdm_cut = apply_wdm_cut(Y_wdm, cut_info)
    Z_wdm_cut = apply_wdm_cut(Z_wdm, cut_info)

    overlap_cut, mismatch_cut = xyz_mismatch(
        X_td_cut, Y_td_cut, Z_td_cut,
        X_wdm_cut, Y_wdm_cut, Z_wdm_cut,
    )

    result = {
        "alpha": float(alpha),
        "beta": float(beta),
        "left_cut": int(left_cut),
        "right_cut": int(right_cut),
        "rows_removed": int(rows_removed),
        "XYZ_mismatch_uncut": float(mismatch_uncut),
        "XYZ_mismatch_cut": float(mismatch_cut),
    }

    print("Result:")
    print(f"  left/right cut:       {left_cut} / {right_cut}")
    print(f"  rows removed:         {rows_removed}")
    print(f"  XYZ mismatch uncut:   {mismatch_uncut:.6e}")
    print(f"  XYZ mismatch cut:     {mismatch_cut:.6e}")

    # Clean large arrays before next run.
    del (
        times_pad, hplus_td, hcross_td,
        td_geom_precomp, td_wave_precomp,
        X_td, Y_td, Z_td,
        X_wdm, Y_wdm, Z_wdm,
        X_td_w, Y_td_w, Z_td_w,
        X_td_cut, Y_td_cut, Z_td_cut,
        X_wdm_cut, Y_wdm_cut, Z_wdm_cut,
        wdm_ref,
        tdi_shift_cache, td_one_arm_cache, wdm_one_arm_cache,
    )
    gc.collect()

    return result

In [6]:
# -------------------------------------------------------------------
# Run alpha/beta sweep
# -------------------------------------------------------------------

results = []

partial_path = OUTPUT_DIR / "alpha_beta_sensitivity_partial.csv"
final_path = OUTPUT_DIR / "alpha_beta_sensitivity_full.csv"

for alpha in alpha_values:
    for beta in beta_values:
        result = run_alpha_beta_case(alpha, beta)
        results.append(result)

        pd.DataFrame(results).to_csv(partial_path, index=False)
        print("Saved partial table to:", partial_path)

df = pd.DataFrame(results)
df = df.sort_values(["alpha", "beta"]).reset_index(drop=True)

df.to_csv(final_path, index=False)

print()
print("Finished sweep.")
print("Saved final table to:", final_path)

df


Running alpha=0.0025, beta=2.0
Waveform generated:
  original length: 315581
  padded length:   318000
  WDM Nt:          636
Result:
  left/right cut:       3 / 5
  rows removed:         8
  XYZ mismatch uncut:   2.940146e-05
  XYZ mismatch cut:     7.610330e-06
Saved partial table to: runs/alpha_beta_boundary_sensitivity_0p1yr/alpha_beta_sensitivity_partial.csv

Running alpha=0.0025, beta=4.0
Waveform generated:
  original length: 315581
  padded length:   319000
  WDM Nt:          638
Result:
  left/right cut:       4 / 5
  rows removed:         9
  XYZ mismatch uncut:   1.540456e-05
  XYZ mismatch cut:     6.959886e-06
Saved partial table to: runs/alpha_beta_boundary_sensitivity_0p1yr/alpha_beta_sensitivity_partial.csv

Running alpha=0.005, beta=2.0
Waveform generated:
  original length: 315581
  padded length:   319000
  WDM Nt:          638
Result:
  left/right cut:       5 / 6
  rows removed:         11
  XYZ mismatch uncut:   1.152705e-05
  XYZ mismatch cut:     6.809823e-06
S

,alpha,beta,left_cut,right_cut,rows_removed,XYZ_mismatch_uncut,XYZ_mismatch_cut
0,0.0025,2.0,3,5,8,0.000029,0.000008
1,0.0025,4.0,4,5,9,0.000015,0.000007
2,0.0050,2.0,5,6,11,0.000012,0.000007
3,0.0050,4.0,8,9,17,0.000007,0.000007
4,0.0100,2.0,10,10,20,0.000007,0.000007
5,0.0100,4.0,16,18,34,0.000007,0.000007


In [7]:
# -------------------------------------------------------------------
# Create LaTeX table matching the report layout
# -------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

# Load results if df is not already in memory
if "df" not in globals():
    final_path = OUTPUT_DIR / "alpha_beta_sensitivity_full.csv"
    df = pd.read_csv(final_path)

# Keep and order only the columns needed for the report table
table_df = df[
    [
        "alpha",
        "beta",
        "left_cut",
        "right_cut",
        "XYZ_mismatch_uncut",
        "XYZ_mismatch_cut",
    ]
].copy()

table_df = table_df.sort_values(["alpha", "beta"]).reset_index(drop=True)


def fmt_decimal(x):
    """
    Format alpha/beta values compactly.
    """
    return f"{float(x):g}"


def fmt_sci_latex(x, sig_figs=3):
    """
    Format a number as LaTeX scientific notation:
    6.74e-06 -> $6.74\\times 10^{-6}$
    """
    x = float(x)

    if x == 0:
        return "$0$"

    exponent = int(np.floor(np.log10(abs(x))))
    mantissa = x / (10 ** exponent)

    return rf"${mantissa:.{sig_figs - 1}f}\times 10^{{{exponent}}}$"


lines = []

lines.append(r"\begin{table}[!h]")
lines.append(r"    \centering")
lines.append(r"    \begin{tabular}{|c|c|c|c|c|c|}")
lines.append(r"    \hline")
lines.append(
    r"        $\alpha$ & $\beta$ & $n_{\mathrm{left}}$ & $n_{\mathrm{right}}$ & "
    r"$\mathcal{M}^{\mathrm{uncut}}_{XYZ}$ & $\mathcal{M}^{\mathrm{cut}}_{XYZ}$  \\"
)
lines.append(r"    \hline")

for _, row in table_df.iterrows():
    line = (
        f"        {fmt_decimal(row['alpha'])} & "
        f"{fmt_decimal(row['beta'])} & "
        f"{int(row['left_cut'])} & "
        f"{int(row['right_cut'])} & "
        f"{fmt_sci_latex(row['XYZ_mismatch_uncut'])} & "
        f"{fmt_sci_latex(row['XYZ_mismatch_cut'])}  \\\\"
    )
    lines.append(line)
    lines.append(r"    \hline")

lines.append(r"    \end{tabular}")
lines.append(
    r"    \caption{Sensitivity of the short-segment TD/WDM validation to the "
    r"Tukey taper width $\alpha$ and zero-padding factor $\beta$. The mismatch "
    r"is computed for the combined $XYZ$ response by treating the three WDM "
    r"coefficient arrays as a single concatenated vector.}"
)
lines.append(r"    \label{tab:alpha-beta-sensitivity}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)

# Save and print
latex_table_path = OUTPUT_DIR / "alpha_beta_sensitivity_table_report.tex"

with open(latex_table_path, "w") as f:
    f.write(latex_table)

print("Saved LaTeX table to:", latex_table_path)
print()
print(latex_table)

Saved LaTeX table to: runs/alpha_beta_boundary_sensitivity_0p1yr/alpha_beta_sensitivity_table_report.tex

\begin{table}[!h]
    \centering
    \begin{tabular}{|c|c|c|c|c|c|}
    \hline
        $\alpha$ & $\beta$ & $n_{\mathrm{left}}$ & $n_{\mathrm{right}}$ & $\mathcal{M}^{\mathrm{uncut}}_{XYZ}$ & $\mathcal{M}^{\mathrm{cut}}_{XYZ}$  \\
    \hline
        0.0025 & 2 & 3 & 5 & $2.94\times 10^{-5}$ & $7.61\times 10^{-6}$  \\
    \hline
        0.0025 & 4 & 4 & 5 & $1.54\times 10^{-5}$ & $6.96\times 10^{-6}$  \\
    \hline
        0.005 & 2 & 5 & 6 & $1.15\times 10^{-5}$ & $6.81\times 10^{-6}$  \\
    \hline
        0.005 & 4 & 8 & 9 & $6.93\times 10^{-6}$ & $6.74\times 10^{-6}$  \\
    \hline
        0.01 & 2 & 10 & 10 & $6.79\times 10^{-6}$ & $6.69\times 10^{-6}$  \\
    \hline
        0.01 & 4 & 16 & 18 & $6.74\times 10^{-6}$ & $6.70\times 10^{-6}$  \\
    \hline
    \end{tabular}
    \caption{Sensitivity of the short-segment TD/WDM validation to the Tukey taper width $\alpha$ and zero-p